#PYTHON EDA FOR KPI METRICS

In [1]:
import pandas as pd 
import sqlalchemy
import pyodbc
from sqlalchemy import create_engine
import urllib

print('sqlalchemy version: ', sqlalchemy.__version__)
print(pyodbc.drivers())

sqlalchemy version:  2.0.39
['SQL Server', 'Devart ODBC Driver for SQLite', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [ ]:
# Server Credentials for pyodbc connection

server       = r'<server_name>'
driver       = 'ODBC Driver 18 for SQL Server'
database     = '<database_name>'

connection_string = (
    f"Driver={{{driver}}};"
    f"Server={server};"
    f"Database={database};"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Connection check
try:
    conn = pyodbc.connect(connection_string)
    conn.close()
    print("✅ Connection successful.")
except pyodbc.Error as e:
    print(f"❌ Connection failed: {e}")

✅ Connection successful.


In [3]:
# sqlalchemy connections

sqla_connection_string = urllib.parse.quote(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")


In [ ]:
# The KPIs will be defined by manufacter: HAUSBERG & WEISSTECH

first_query_hau = """SELECT 
       [source_id] 
      ,[quality]
      ,[performance]
      ,[availability]
      ,[pplh]
      ,[ftq]
      ,[oee]
  FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
  WHERE source_id LIKE '%HAUSBERG%'
"""
df_status_hau = pd.read_sql(first_query_hau, engine)
df_status_hau.describe()

c:\Users\amcys\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


,quality,performance,availability,pplh,ftq,oee
count,163603.000000,163603.000000,163603.000000,163603.000000,163603.000000,163603.000000
mean,0.963564,0.895810,0.809305,0.331133,0.943570,0.688606
std,0.033856,0.139672,0.093739,0.170897,0.046006,0.076624
min,0.714300,0.373200,0.457100,0.067200,0.594600,0.310800
25%,0.949200,0.830700,0.742900,0.182100,0.921600,0.639000
50%,0.973300,0.911300,0.800000,0.263200,0.957100,0.691200
75%,0.986300,0.992400,0.871400,0.468400,0.977400,0.740650
max,1.000000,1.149600,1.000000,0.834600,1.000000,0.999400


In [ ]:
# The KPIs will be defined by manufacter: WEISSTECH & HAUSBERG

first_query_wei = """SELECT 
       [source_id] 
      ,[quality]
      ,[performance]
      ,[availability]
      ,[pplh]
      ,[ftq]
      ,[oee]
  FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
  WHERE source_id LIKE '%WEISSTECH%'
"""
df_status_weipd.read_sql(first_query_wei, engine)
df_status_weiscribe()

,quality,performance,availability,pplh,ftq,oee
count,164044.000000,164044.000000,164044.000000,164044.000000,164044.000000,164044.000000
mean,0.960615,0.895217,0.809428,0.279251,0.939066,0.686102
std,0.033843,0.139902,0.093716,0.125909,0.045202,0.076183
min,0.729700,0.437900,0.450700,0.079300,0.634100,0.338400
25%,0.944400,0.829600,0.743200,0.181800,0.915700,0.637000
50%,0.968400,0.910250,0.802300,0.238400,0.949200,0.688600
75%,0.984100,0.991900,0.871400,0.355400,0.972200,0.737725
max,1.000000,1.149600,1.000000,0.829700,1.000000,0.982900


In [ ]:
# Checking total rows of data 
total_rows= """
SELECT
SUM(total_rows) AS total_rows_sum
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub].[gold_layer].[fact_status_table_final_dev]
    GROUP BY source_id) t
"""

df_total_rows = pd.read_sql(total_rows, engine)
df_total_rows

,total_rows_sum
0,327647


##Checking how many records we have of both WEISSTECH and HAUSBERG cars where the production surpassed 85% of oee 

In [ ]:
# Total rows of HAUSBERG
total_rows_hau = """
SELECT
SUM(total_rows) AS total_rows_sum_hau
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse_pub.[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%' AND oee >= 0.85
    GROUP BY source_id) t
"""

df_total_rows_hau = pd.read_sql(total_rows_hau, engine)
df_total_rows_hau

,total_rows_sum_bmw
0,2449


In [ ]:
# Total rows of hau
total_rows_hau = """
SELECT
SUM(total_rows) AS total_rows_sum
FROM
    (SELECT 
          [source_id],
          COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%'
    GROUP BY source_id) t
"""

# Total rows of hau products above 85% OEE
total_rows_sum_hau = """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%HAUSBERG%' AND oee >= 0.85
    GROUP BY [source_id]
) t
"""

df_total_rows_hau = pd.read_sql(total_rows_hau, engine)
df_total_rows_hau_85 = pd.read_sql(total_rows_sum_hau, engine)

denom = df_total_rows_hau.iloc[0, 0]
percentage_hau = df_total_rows_hau_85.iloc[0, 0] / denom

print(f"total rows of HAUSBERG production: {df_total_rows_hau.iloc[0, 0]:,.0f}")
print(f"total rows of HAUSBERG production above 85% OEE: {df_total_rows_hau_85.iloc[0, 0]:,.0f}")
print(f"% of rows with more than 85% oee: {percentage_hau:.2%}")

total rows of BMW production: 163,603
total rows of BMW production above 85% OEE: 2,449
% of rows with more than 85% oee: 1.50%


In [ ]:
import pandas as pd

# Total rows of WEISSTECH
total_rows_wei= """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%WEISSTECH%'
    GROUP BY source_id
) t
"""

# Total rows of WEISSTECH above 85% OEE
total_rows_sum_wei = """
SELECT SUM(total_rows) AS total_rows_sum
FROM (
    SELECT [source_id], [oee], COUNT(source_id) AS total_rows
    FROM [db_manufacturing_warehouse].[gold_layer].[fact_status_table_final_dev]
    WHERE source_id LIKE '%WEISSTECH%' AND [oee] > 0.85
    GROUP BY source_id, oee
) t
"""

df_total_rows_wei = pd.read_sql(total_rows_wei, engine)
df_total_rows_wei_85 = pd.read_sql(total_rows_sum_wei)

denom = df_total_rows_wei.iloc[0, 0]
percentage_wei = df_total_rows_wei_85.iloc[0, 0] / denom if denom else 0

print(f"total rows of WEISSTECH production: {df_total_rows_wei.iloc[0, 0]:,.0f}")
print(f"total rows of WEISSTECH production: {df_total_rows_wei.iloc[0, 0]:,.0f}")
print(f"total rows of WEISSTECH production above 85% OEE: {df_total_rows_wei_85.iloc[0, 0]:,.0f}")
print(f"% of rows with more than 85% oee: {percentage_wei:.2%}")

total rows of Mercedes production: 164,044
total rows of Mercedes production above 85% OEE: 2,101
% of rows with more than 85% oee: 1.28%


## Final Conclusion

In HAUSBERG products, only **2.434** production days got an OEE higher than **85%**, which correspond to **1.50%** of all HAUSBERG status data

In WEISSTECH products, only **2.101** production days got an OEE higher than **85%**, which correspond to **1.28%** of all WEISSTECH status dataimport pandas as pd

Since we have few data to use the golden oee rule of 85%, we'll base ourselves on the third quadrand (75%).

The HAUSBERG KPI goals for green color 🟢 will be:
    OEE: 74.06%
    Performance: 90.0%
    Availability: 83.12% 
    Quality: 99.0%
    FTQ: 97.74%
    PPLH: 0.46

The HAUSBERG KPI goals for yellow color 🟡 will be 90% of the green KPIs. The OEE metric will be 95% of the green one:
    OEE: 70.35%
    Performance: 88.0%
    Availability: 81.57%
    Quality: 98.0%
    FTQ: 92.85%
    PPLH: 0.414

Below the minimum for yellow KPIs, we'll display the color red 🔴
Sometimes the performance surpass the 100% and this is incorrect. To call the user attention we will display it with purple 🟣

Values of reference:
quality	performance	availability	pplh	ftq	oee
count	163603.000000	163603.000000	163603.000000	163603.000000	163603.000000	163603.000000
mean	0.963564	0.895810	0.809305	0.331133	0.943570	0.688606
std	0.033856	0.139672	0.093739	0.170897	0.046006	0.076624
min	0.714300	0.373200	0.457100	0.067200	0.594600	0.310800
25%	0.949200	0.830700	0.742900	0.182100	0.921600	0.639000
50%	0.973300	0.911300	0.800000	0.263200	0.957100	0.691200
75%	0.986300	0.992400	0.871400	0.468400	0.977400	0.740650
max	1.000000	1.149600	1.000000	0.834600	1.000000	0.999400

The WEISSTECH KPI goals for green color 🟢 will be:
    OEE: 73.77%
    Performance: 90.0%
    Availability: 82.79% 
    Quality: 99.0%
    FTQ: 97.74%
    PPLH: 0.46

The WEISSTECH KPI goals for yellow color 🟡 will be 90% of the green KPIs. The OEE metric will be 95% of the green one:
    OEE: 70.08%
    Performance: 88.0%
    Availability: 81.26%
    Quality: 98.0%
    FTQ: 87.96%
    PPLH: 0.414

Below the minimum for yellow KPIs, we'll display the color red 🔴
Sometimes the performance surpass the 100% and this is incorrect. To call the user attention we will display it with purple 🟣

    	quality	performance	availability	pplh	ftq	oee
count	164044.000000	164044.000000	164044.000000	164044.000000	164044.000000	164044.000000
mean	0.960615	0.895217	0.809428	0.279251	0.939066	0.686102
std	0.033843	0.139902	0.093716	0.125909	0.045202	0.076183
min	0.729700	0.437900	0.450700	0.079300	0.634100	0.338400
25%	0.944400	0.829600	0.743200	0.181800	0.915700	0.637000
50%	0.968400	0.910250	0.802300	0.238400	0.949200	0.688600
75%	0.984100	0.991900	0.871400	0.355400	0.972200	0.737725
max	1.000000	1.149600	1.000000	0.829700	1.000000	0.982900
